# Day 3: ETL - Extract & Transform with Pandas

Today we bridge your existing `pandas` skills with your new `SQL` skills. We'll focus on the 'E' (Extract) and 'T' (Transform) parts of the ETL (Extract, Transform, Load) process.

### 1. Setup and Imports

In [1]:
import pandas as pd
import sqlite3
import numpy as np

### 2. Core Concept: What is ETL?

- **Extract**: Get data from a source. This could be a CSV file, a website (API), or another database.
- **Transform**: Clean the data. This is where you use `pandas`! You'll fix missing values, change data types, create new columns, merge datasets, etc.
- **Load**: Put the clean, transformed data into its final destination. This is often a data warehouse or, for our purposes, a new table in our SQLite database.

Today, we focus on **Extract** and **Transform**.

### 3. (E) Task 1: Extract Data

We will practice two common extraction methods.

#### Method 1: Extract from a CSV file

Let's first create a dummy CSV file to simulate a common data source.

In [2]:
raw_data = {
    'employee_id': [101, 102, 103, 104, 105],
    'full_name': ['Sarah Jenkins', 'Mike T.', 'Leo Kim', 'Anjali Gupta', 'Mark O\'Brien'],
    'job_title': ['Data Scientist', 'sr. analyst', 'data engineer', 'Data Scientist', 'Product Manager'],
    'hire_date': ['2021-05-10', '2020-03-15', '2022-01-20', '2021-05-10', '2019-11-30'],
    'salary_str': ['$120,000', '$85,000', '130000', '$125,000', '$110,000']
}

df_raw = pd.DataFrame(raw_data)
df_raw.to_csv('raw_hr_data.csv', index=False)

print("Created 'raw_hr_data.csv'")

Created 'raw_hr_data.csv'


In [3]:
df = pd.read_csv('raw_hr_data.csv')

print("Data Extracted from CSV:")
df.head()

Data Extracted from CSV:


,employee_id,full_name,job_title,hire_date,salary_str
0,101,Sarah Jenkins,Data Scientist,2021-05-10,"$120,000"
1,102,Mike T.,sr. analyst,2020-03-15,"$85,000"
2,103,Leo Kim,data engineer,2022-01-20,130000
3,104,Anjali Gupta,Data Scientist,2021-05-10,"$125,000"
4,105,Mark O'Brien,Product Manager,2019-11-30,"$110,000"


#### Method 2: Extract from our SQL Database

This is where `pandas` and `SQL` meet! `pd.read_sql_query()` is one of the most useful functions you'll learn. It runs a SQL query and returns the result *directly* into a pandas DataFrame.

In [4]:
conn = sqlite3.connect('db/company.db')

# This is our query from yesterday
query = """
SELECT e.name, d.dept_name, e.salary
FROM employees AS e
INNER JOIN departments AS d ON e.dept_id = d.dept_id
"""

df_sql = pd.read_sql_query(query, conn)

conn.close()

print("Data Extracted from SQL DB:")
df_sql.head()

Data Extracted from SQL DB:


,name,dept_name,salary
0,Alice Smith,Engineering,95000.0
1,Bob Johnson,Sales,65000.0
2,Charlie Brown,Engineering,110000.0
3,David Lee,Sales,70000.0


### 4. (T) Task 2: Transform Data

Let's use the messy CSV data (`df`) and clean it up using our `pandas` skills. This is the **Transform** step.

**Our To-Do List:**
1.  `salary_str` is a string with '$' and ','. We need to convert it to a number (`REAL` or `INTEGER`).
2.  `job_title` is inconsistent ('Data Scientist' vs 'data engineer'). Let's make it consistent (e.g., Title Case).
3.  `hire_date` is a string. Let's convert it to a proper `datetime` object.
4.  `full_name` has first and last names. Let's split it into `first_name` and `last_name` columns.

In [6]:
print("Transforming data...")
df_transformed = df.copy()

# 1. Clean salary_str
df_transformed['salary'] = df_transformed['salary_str'].replace(r'[$,]', '', regex=True).astype(float)

# 2. Clean job_title
df_transformed['job_title'] = df_transformed['job_title'].str.title()

# 3. Convert hire_date
df_transformed['hire_date'] = pd.to_datetime(df_transformed['hire_date'])

# 4. Split full_name
df_transformed[['first_name', 'last_name']] = df_transformed['full_name'].str.split(' ', n=1, expand=True)

df_transformed = df_transformed.drop(['full_name', 'salary_str'], axis=1)

print("Data Transformed!")
df_transformed.head()

Transforming data...
Data Transformed!


,employee_id,job_title,hire_date,salary,first_name,last_name
0,101,Data Scientist,2021-05-10,120000.0,Sarah,Jenkins
1,102,Sr. Analyst,2020-03-15,85000.0,Mike,T.
2,103,Data Engineer,2022-01-20,130000.0,Leo,Kim
3,104,Data Scientist,2021-05-10,125000.0,Anjali,Gupta
4,105,Product Manager,2019-11-30,110000.0,Mark,O'Brien


In [7]:
# Let's check our data types to confirm the transformations
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   employee_id  5 non-null      int64         
 1   job_title    5 non-null      object        
 2   hire_date    5 non-null      datetime64[ns]
 3   salary       5 non-null      float64       
 4   first_name   5 non-null      object        
 5   last_name    5 non-null      object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 372.0+ bytes


### Day 3 Summary

Excellent! You have now mastered the 'E' and 'T' of ETL.

You learned how to:
1.  **Extract** data from a flat file (`.csv`).
2.  **Extract** data from a SQL database *directly* into a DataFrame using `pd.read_sql_query()`.
3.  **Transform** that data using your `pandas` skills (cleaning strings, converting types, creating new columns).

Tomorrow is the final step: **(L) Load**. We will take our clean `df_transformed` and load it into a new, clean table in our SQLite database.